# 状態予測モデル

状態予測モデルでは、未来予測を使った意思決定を、実装可能な形に分解して学びます。


## 当たる予測と、使える予測は同じではない

状態予測モデルは、次に何が起こるかを先回りして見るための土台です。ただし 1 ステップ先がそこそこ当たるだけでは不十分で、その予測を何歩も先までつなげたときに崩れないかまで見ないと計画には使えません。ここでいうロールアウトとは、モデルの予測結果を次の材料として何度も使いながら未来を順にたどることです。最初は「予測を1回で終わらせず、その結果をまた次の予測へ回すこと」だと思えば十分です。


この notebook では、未来状態の予測をそのまま異常検知にもつなげて考えます。予測が外れた瞬間は、単なる失敗ではなく『いつもと違う』を見つける手掛かりにもなるからです。


## 1 ステップ誤差と多ステップ誤差を分けて読む

次の一歩だけなら当たるモデルでも、その予測を自分で食べながら先へ進むとすぐ崩れることがあります。ここで見たいのは、誤差の大きさそのものより、どのくらいの長さから累積が目立ち始めるかです。言い換えると、「1回だけ使うなら便利」なのか、「何回つないでもまだ使える」なのかを分けて見ています。


## この notebook の見どころ

初期遷移の確認、観測復元との役割分担、何歩先から予測が崩れ始めるか、複数の行動案の比べ方、最後に誤差監視までを一続きで見ます。


異常検知まで視野に入れるなら、平均誤差だけでは足りません。どの時点で、どの条件で、どの誤差が急に大きくなるかを見る必要があります。


## 誤差が意味を持つ瞬間

予測誤差は、モデル改善の手掛かりであると同時に運用上の警報でもあります。ずれが大きくなった時点を丁寧に見ると、モデルの弱い領域と異常候補の両方が見えてきます。


## この notebook の立ち位置

ここでのロールアウトは小さな教材用の例ですが、1 ステップ評価と長期評価を分ける考え方は実務でもそのまま重要です。短く当たるだけのモデルと、先まで試しても使えるモデルは別物だからです。


## 実験 1: 状態予測モデルの初期遷移

未来状態予測の誤差を見るため、まずは『いま内部でどんな状態にあると考えるか』と『そこへどんな操作を加えるか』を数で置きます。ここでの遷移とは、その内部状態が 1 歩先でどう変わるかを表すルールです。


In [ ]:
z_t = -0.05
a_t = 1.1
A, B = 0.89, 0.21
z_next = A * z_t + B * a_t
print('task = state-prediction-models')
print('z_next =', round(z_next, 6))

この遷移誤差を後続で時系列的に評価します。



## 実験 2: 観測予測を作る

次に、内部状態から外に見える量を復元する写像を作ります。潜在状態とは直接は見えないが、予測の中で持っている要約情報のことです。最初は、潜在状態を「外からは見えないが、モデルが内部で持っている作業メモ」だと思えば十分です。状態推定と観測再現の役割分担をコードで掴みます。


In [ ]:
def decode(z):
    return {'position': 2.5 * z + 0.1, 'velocity': 0.8 * z - 0.05}
obs_next = decode(z_next)
print('obs_next =', {k: round(v, 4) for k, v in obs_next.items()})
print('keys =', list(obs_next.keys()))

観測予測を別関数に切ると、内部状態の進め方が悪いのか、そこから見える量へ戻す部分が悪いのかを分けて調整できます。
つまり、「考え方のメモが悪いのか」「そのメモを見える量へ訳す段階が悪いのか」を別々に直せるようになります。



## 式と実装の往復

1. $z_{t+1} = f_{\theta}(z_t, a_t)$
2. $\hat{o}_{t+1} = g_{\theta}(z_{t+1})$


## 実験 3: ロールアウトを試す

ここで複数ステップ予測を実行します。1 ステップでは見えない誤差累積を把握するためです。未来を 1 回だけ当てるのでなく、その結果をまた次の予測へ渡して先まで進めます。ここで見たいのは、「先の予測が外れるか」だけでなく、「どの時点から外れ始めるか」です。


In [ ]:
actions = [0.0, 1.0, 1.0, 0.0, -0.5]
z = 0.1
traj = []
for a in actions:
    z = 0.92 * z + 0.18 * a
    traj.append(round(z, 5))
print('rollout =', traj)

長期予測で崩れるなら、内部状態の持ち方が粗いか、1 歩進めるルールが少しずつずれている可能性を疑います。
1 歩ずつは小さなズレでも、何回もつなぐと大きな外れになるので、ロールアウトはその増え方を見る検査でもあります。



## 実験 4: 計画候補を比較する

次に、複数の行動列を比較して、どの行動案が望ましいかを評価します。現実で全部試す前に、モデルの中で先に試して比べるのがモデルベース強化学習の中心操作です。要するに、「本番で動く前に、頭の中で試走して比べる」段階です。


In [ ]:
plans = [[0, 1, 1], [1, 1, 1], [0, 0, 1]]
def score_plan(plan):
    z = 0.1
    for a in plan:
        z = 0.92 * z + 0.18 * a
    return z
scores = [round(score_plan(p), 5) for p in plans]
print('scores =', scores)

計画評価が可能になると、実環境での試行回数を抑えながら『まず試す価値が高そうな案』を選びやすくなります。
ここでは score を厳密な報酬とみなすより、「どの案が目的に近そうかを並べ替える代理点」くらいに読むと十分です。



## 実験 5: モデル誤差を監視する

最後に、予測と実測の差を定量化します。世界モデルは『予測できる範囲』を常に点検する運用が重要です。


In [ ]:
pred = [0.10, 0.22, 0.31, 0.29]
real = [0.11, 0.25, 0.28, 0.35]
errors = [abs(p - r) for p, r in zip(pred, real)]
print('errors =', [round(e, 4) for e in errors])
print('mean_error =', round(sum(errors) / len(errors), 5))

平均誤差だけでなく時点別誤差を追うと、どの条件からモデルが外れやすくなるかを特定しやすくなります。異常検知に使うなら、たとえば `mean_error` の平均だけを見るのでなく、各時点誤差の 95 パーセンタイルや固定閾値を超えた瞬間を異常候補とみなします。


## 読み終えたあとに残したい視点

1. 1 ステップ先の正確さだけではモデルの価値は決まらない。
2. 何歩も先までつないだ予測は、誤差がどこで積み上がるかの見取り図になる。
3. 予測誤差は、異常検知の入口としても読む価値がある。
